# 02 - Run the worker and see its trail in Azure

**Goal:** run a real Agent Framework agent with the protected MCP tool, then find the same execution in Azure Application Insights.

**Before starting:** finish [01 - Agent permissions](./01_Agent_Permissions.ipynb). Keep the same blueprint, child, MCP API, and `Workshop.Read.Application` app-role grant. **Restart the kernel** and run this notebook from the top; tracing is configured once per kernel.

A **span** records one action, like a tool call. A **trace** joins the actions for one job, like a receipt with several line items. We will use its trace ID to find it in Azure.

**This lesson does not need Agent 365 registration or licensing.** Azure Application Insights is a separate destination. We add Agent 365 in notebook 04.

## 1. Connect the model - one-time setup, then sign in each session

**An Azure administrator:** supplies an existing Azure OpenAI deployment and grants your CLI user **Cognitive Services OpenAI User** on its resource. The endpoint must be reachable from this computer. Calls can cost money.

**You:** save these values in the private `.env`:

```dotenv
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com
AZURE_OPENAI_DEPLOYMENT=<existing-deployment-name>
AZURE_OPENAI_API_VERSION=2025-01-01-preview
```

Sign in to Azure CLI in that tenant before the demo. If Azure CLI is not installed, install it first. VS Code account sign-in alone is not CLI sign-in.

**Whose permission?** The model connection uses **your Azure CLI user**. The protected MCP calls use **the child agent identity**. We are not claiming every operation runs as the child.

## 2. Pick the Azure trace destination - once

**An Azure administrator:** reuse an Application Insights resource or, with approval, create one in the Azure portal (**Create a resource > Application Insights**). Choose the subscription, resource group, region, and Log Analytics workspace. Ingestion/storage can cost money; this notebook creates nothing.

1. Open **Application Insights > Overview > Connection String**. Copy the whole connection string privately into `.env` as `APPLICATIONINSIGHTS_CONNECTION_STRING`.
2. Give the CLI user **Monitoring Metrics Publisher** on that Application Insights resource. Our exporter uses this user to authenticate ingestion, even though spans describe the agent.
3. Give the viewer query access, for example **Monitoring Reader** on the resource and **Log Analytics Reader** on the linked workspace, subject to your organization's access model.
4. Make sure Azure Monitor ingestion/query endpoints are reachable; private-only resources need the approved network connection.

**Expected:** you can open this exact resource's **Logs** page. Do not look in Entra for execution traces. Entra only has identity/sign-in records.

**Privacy:** we disable prompt/completion capture and offline trace storage. Tool names, timing, identifiers, and error information are still telemetry. In this lesson only read-only identity results go to the model; no tokens or secrets.

## 3. Load settings - each fresh kernel

**Setup needed:** notebooks 00-01, steps 1-2 above, and the project kernel with `requirements-showcase.txt` installed. Start from the project root or `Agent_365`.

This loads settings from `.env` without printing credentials.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parent if Path.cwd().name == 'Agent_365' else Path.cwd()
sys.path.insert(0, str(root))
from Agent_365.mcp_workshop import MCPSettings, sign_in_agent, run_mcp_assistant
from Agent_365.workshop_helpers import start_recording, show_activities
from Agent_365.workshop_tracing import configure_workshop_tracing, flush_workshop_trace

settings = MCPSettings.from_env()

## 4. Turn on the recorder, then authenticate the child

**Setup needed:** step 3 succeeded. Run the tracing configuration only once per kernel; restart rather than rerunning it.

`configure_workshop_tracing` adds Azure export and a local copy of the same spans, then enables Agent Framework's instrumentation. It does not turn on Agent 365. `sign_in_agent` separately obtains the child's signed MCP token, exactly as in notebook 01.

In [ ]:
configure_workshop_tracing(settings.tenant_id, settings.agent_id)
access = await sign_in_agent(settings)

## 5. Let the agent use the protected tool

**Setup needed:** step 4 succeeded; the CLI user can call the model. No new setup.

Inside `run_mcp_assistant`, the important wiring is:

```python
httpx.AsyncClient(headers=access.headers())        # sends the child's token to MCP
MCPStreamableHTTPTool(..., http_client=http)       # discovers only authorized tools
Agent(id=settings.agent_id, ..., tools=[mcp_tools])
await assistant.run(...)                         # runs the model/tool conversation
```

The helper handles connection cleanup. The server checks the token **before** the model sees the tool list and checks permission again on each call. `Agent(id=...)` labels the runtime; the token, not the label, proves identity.

```text
Your CLI user -> Azure model
Local Agent Framework agent -> child's signed token -> protected MCP tool
Agent/model/tool spans -> Azure Application Insights + local activity table
```

In [ ]:
start_recording()
await run_mcp_assistant(settings, access)
show_activities()
trace_id = await flush_workshop_trace()

## 6. Find the exact run in Azure - the proof step

**Setup needed:** step 5 completed; query permissions from step 2. A flushed queue is not proof of ingestion.

1. Open the **same Application Insights resource** whose connection string you configured.
2. Select **Logs**, switch to **KQL mode** if needed, and paste the query printed above. It uses this run's exact trace ID.
3. Allow a few minutes for ingestion. Keep the portal time range wide enough for this run.
4. Find the agent invocation, model calls, and tool execution. In **Transaction search / end-to-end transaction details**, inspect their parent/child relationship if that view is available.

The printed query uses Application Insights table names `requests` and `dependencies`. If querying the linked Log Analytics workspace instead, use `AppRequests` / `AppDependencies` and columns `TimeGenerated`, `OperationId`, `ParentId`, `Name`, and `Id`.

**If there are no rows:** check exporter errors, CLI sign-in, **Monitoring Metrics Publisher**, the destination connection string, network access, and the query time range. Do not call this a successful cloud demo until the rows appear.

**Say:** "The local agent used its Entra badge for MCP. This Azure trace shows what the worker did. It does not prove Agent 365 onboarding."

Next: [03 - User delegation](./03_User_Delegation.ipynb), or [04 - Add Agent 365](./04_Agent365_Observability.ipynb).